# [ICCAD 2025] SROA on Attar: GPU detailed placement for quality, step by step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RayChang987/Attar/blob/main/examples/colab/sroa_on_attar.ipynb)

**SROA** (*Simultaneous Row and Order Assignment*; Lu, Liu, Ren, Wang, "Leveraging GPU for Better Detailed
Placement Quality", ICCAD 2025) uses the GPU not to make detailed placement faster but to search a much larger
space than CPU detailed placers can afford:

1. take n neighbouring cells (4 to 6) spread over m adjacent rows (2 or 3): a **task**;
2. try **every permutation** of them (720 for 6 cells), one GPU thread each;
3. for each permutation, a **dynamic program** decides which row every cell goes to, and a heuristic its x;
4. keep the best permutation if it shortens the wires; tasks that share no net run together.

The flow runs twice: first minimizing **HPWL** (to shrink the nets' boxes), then the **rectilinear MST length**
(closer to the routed wirelength), each stage preceded by independent set matching (ISM).

[Attar](https://github.com/RayChang987/Attar) supplies everything around the algorithm: the ISPD 2019 design,
rows, sites, power rails and PDN rules, the batched cost of candidate moves, legality checks and a global router
to judge the result. What follows is only SROA and ISM, split into the pieces of the paper.

**Runtime:** use a GPU runtime (T4). The whole notebook on `ispd19_test1` (8.9k cells) takes 45 s on an RTX 4090
and a few minutes on a T4.

## 0. Setup

In [ ]:
!pip install -q attar==1.0.6

In [ ]:
import itertools
import os
import tempfile
import time

import numpy as np
import torch
from scipy.optimize import linear_sum_assignment

import attar
from attar import Evaluator, naive
from attar import metrics as M
from attar.design import _orient_offset
from attar.naive.dp import _DP

INF = float('inf')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('attar', attar.__version__, '| torch', torch.__version__, '| device', DEVICE)
try:                                   # Triton (with CUDA torch): the GPU MST kernel of section 4
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:                    # CPU-only runtime: a stand-in so the kernel's definition still parses
    HAS_TRITON = False

    class triton:
        jit = staticmethod(lambda f: f)

    class tl:
        constexpr = None

## 1. The design, the ruler and the starting placement

`ispd19_test1` (8.9k cells, 32 nm) from the ICCAD 2019 contest, Problem C, as in the paper; Attar downloads it
from its zoo on first use.

**The ruler.** The paper routes each result with Innovus and reports the detailed-routing wirelength and vias.
Here every placement is judged by **Attar's global router** (`d.global_route()`: wirelength and vias), with HPWL
and the MST wirelength of the nets SROA optimizes (2 to 99 pins, as the detailed placers skip larger nets). On the
GPU the MST is a small Triton kernel (Prim's algorithm, one program per net, double precision); elsewhere Attar's
`metrics.rmst`.

In [ ]:
@triton.jit
def _prim_kernel(q_p, n_p, out_p, D: tl.constexpr):
    b = tl.program_id(0)
    lane = tl.arange(0, D)
    n = tl.load(n_p + b)
    valid = lane < n
    x = tl.load(q_p + (b * D + lane) * 2, mask=valid, other=0.0)
    y = tl.load(q_p + (b * D + lane) * 2 + 1, mask=valid, other=0.0)
    x0 = tl.sum(tl.where(lane == 0, x, 0.0))
    y0 = tl.sum(tl.where(lane == 0, y, 0.0))
    BIG = 1e300
    key = tl.where(valid, tl.abs(x - x0) + tl.abs(y - y0), BIG)
    intree = (lane == 0) | ~valid
    key = tl.where(intree, BIG, key)
    tot = tl.sum(tl.where(lane == 0, x, 0.0)) * 0.0               # float64 zero
    for _ in range(D - 1):
        kmin = tl.min(key, 0)
        j = tl.min(tl.where(key == kmin, lane, D), 0)            # lowest lane of the minimum
        use = kmin < BIG
        tot += tl.where(use, kmin, 0.0).to(tl.float64)
        xj = tl.sum(tl.where(lane == j, x, 0.0))
        yj = tl.sum(tl.where(lane == j, y, 0.0))
        intree = intree | (lane == j)
        key = tl.minimum(key, tl.abs(x - xj) + tl.abs(y - yj))
        key = tl.where(intree, BIG, key)
    tl.store(out_p + b, tot)


def prim_bucket(P, ptr, deg, nets, D):
    """RMST length (float64) of `nets` (degrees <= D, a power of two), pins P (float64, grouped by net, ptr)."""
    dev = P.device
    k = torch.arange(D, device=dev)
    dn = deg[nets]
    idx = ptr[nets][:, None] + torch.where(k[None, :] < dn[:, None], k[None, :], torch.zeros_like(k)[None, :])
    q = P[idx].to(torch.float64).contiguous()                      # (n, D, 2)
    out = torch.empty(len(nets), dtype=torch.float64, device=dev)
    _prim_kernel[(len(nets),)](q, dn.to(torch.int32).contiguous(), out, D=D)
    return out


RMST_BUCKETS = ((4, 8), (9, 16), (17, 32), (33, 64), (65, 128))
prim_bucket = prim_bucket if HAS_TRITON and DEVICE == 'cuda' else None


def rmst(P, deg, ptr, which):
    """Rectilinear MST length (float64) of the nets in `which` (degrees 4..128), pins P (grouped by net, ptr):
    Prim, one Triton program per net on the GPU (attar.metrics.rmst on the CPU). Others: +inf."""
    E = len(deg)
    out = torch.full((E,), INF, dtype=torch.float64, device=P.device)
    if P.device.type != 'cuda' or prim_bucket is None:
        r = M.rmst(P, deg, ptr, E).to(torch.float64)
        return torch.where(which, r, out)
    for dlo, dhi in RMST_BUCKETS:
        nets = torch.nonzero(which & (deg >= dlo) & (deg <= dhi)).flatten()
        if len(nets):
            out[nets] = prim_bucket(P, ptr, deg, nets, dhi)
    return out


def hpwl_costed(d, S):
    """Weighted HPWL of the costed nets (2..99 pins), um."""
    dev = d.device
    P = d.pos.double()[d.p_cell] + d.size.double()[d.p_cell] / 2 + d.pin_off.double()
    E = len(S.deg)
    lo = torch.full((E, 2), INF, dtype=torch.float64, device=dev).scatter_reduce(0, d.p_net[:, None].expand(-1, 2), P, 'amin')
    hi = torch.full((E, 2), -INF, dtype=torch.float64, device=dev).scatter_reduce(0, d.p_net[:, None].expand(-1, 2), P, 'amax')
    hp = torch.where(S.net_ok, (hi - lo).sum(1), torch.zeros(E, dtype=torch.float64, device=dev))
    return float((hp * S.w_net).sum())


def mstwl(d, S=None):
    """Design MSTWL: per costed net (2..99 pins) the RMST (HPWL for 2-3 pins), weighted, um."""
    S = S or _DP(d)
    dev = d.device
    pos = d.pos.double() + d.size.double() / 2
    P = pos[d.p_cell] + d.pin_off.double()
    E = len(S.deg)
    ok = S.net_ok
    lo = torch.full((E, 2), INF, dtype=torch.float64, device=dev).scatter_reduce(0, d.p_net[:, None].expand(-1, 2), P, 'amin')
    hi = torch.full((E, 2), -INF, dtype=torch.float64, device=dev).scatter_reduce(0, d.p_net[:, None].expand(-1, 2), P, 'amax')
    hp = torch.where(ok, (hi - lo).sum(1), torch.zeros(E, dtype=torch.float64, device=dev))
    order = torch.argsort(d.p_net, stable=True)
    r = rmst(P[order], S.deg, d.n_ptr, ok & (S.deg >= 4))
    wl = torch.where(ok & (S.deg >= 4), r, hp)
    return float((wl * S.w_net).sum())


def measure(d, S=None, gr=True):
    """HPWL (all nets), costed HPWL and MSTWL (2..99 pins), GR wirelength / vias / overflow, legality."""
    ev = Evaluator(d)
    out = {'hpwl': float(ev.hpwl())}
    if S is not None:
        out['hpwl_costed'] = hpwl_costed(d, S)
        out['mstwl'] = mstwl(d, S)
    L = ev.legality()
    out['legal'] = bool(L['legal'])
    if gr:
        t = time.time()
        _, st = d.global_route(verbose=False)
        out.update(gr_wl=float(st['wirelength_um']), gr_vias=int(st['vias']), gr_ovf=float(st['overflow_3d']),
                   gr_s=time.time() - t)
    return out

**The start.** The paper starts from DREAMPlace's global placement and legalization. Here Attar places and
legalizes the design itself (`naive.place`, `naive.legalize`), once; the result is saved as a DEF so that every
placer below starts from exactly the same placement.

In [ ]:
START = os.path.join(tempfile.mkdtemp(), 'start.def')
d = attar.load('after_fp/ispd2019/test1', device=DEVICE)
t = time.time()
naive.place(d, verbose=False)
naive.legalize(d, verbose=False)
print('Attar GP + LG: %.0f s' % (time.time() - t))
d.export(START)


def placed():
    """A fresh ispd19_test1 at the start placement."""
    dd = attar.load('after_fp/ispd2019/test1', device=DEVICE)
    dd.read_placement(START)
    return dd


start = measure(d, _DP(d))
print('start:', {k: round(v, 1) for k, v in start.items() if k != 'gr_s'})

## 2. SROA tasks (paper III-B)

Rows from `rof` on are grouped `m` at a time. In each group the cells are sorted by x, the first `cof` skipped,
and every `n` consecutive cells form a task. Because a task's cells are consecutive in that order, its cells in
one row are neighbours in that row; together with the free sites to their right (their **cell spaces**) they
make the row's **available space** for the task (paper Fig. 5). `rof` and `cof` change which cells meet in a task.

Attar's `_DP` (its own detailed placer's view of the design) gives each cell's row, site and free row segment.

In [ ]:
class SROA:
    """SROA passes on a legal placement of design d (the methods are defined in the cells below)."""

def __init__(self, d, K=16, Q=128, top=8, chunk=None, seed=0, verbose=True):
    self.d, self.dev = d, d.device
    self.S = _DP(d)
    S = self.S
    sw = S.sw.unique()
    if len(sw) != 1:
        raise ValueError('rows with different site widths')
    self.sw = float(sw[0])
    self.X0 = float(S.rx0.min())
    ph = ((S.rx0 - self.X0) / self.sw)
    if float((ph - ph.round()).abs().max()) > 1e-6:
        raise ValueError('rows not on one site grid')
    self.roff = ph.round().long()                       # row origin, global sites
    self.K, self.Q, self.top, self.verbose, self.seed = K, Q, top, verbose, seed
    self.dt = torch.float32 if self.dev.type == 'cuda' else torch.float64   # DP only; exact costs are float64
    self.chunk = chunk or (256 if self.dev.type == 'cpu' else 8192)
    self.perms = {}


def tasks(self, n, m, rof, cof):
    """Tasks of one configuration: dict with cells (T, n) design ids in x order, rows (T, m) row ids (-1 past the
    last row), lo / hi (T, m) available space in global sites (lo = hi: none), x (T, n) / w / space in sites."""
    S, dev = self.S, self.dev
    S.refresh()
    ci = torch.arange(len(S.cells), device=dev)
    row, w = S.row, S.w.long()
    gx = S.x.long() + self.roff[row]                     # global site x
    ok = row >= rof
    grp = torch.where(ok, (row - rof) // m, torch.full_like(row, -1))
    ci, grp = ci[ok], grp[ok]
    o = torch.argsort(grp.double() * 1e12 + gx[ci].double() * 8 + row[ci].double() % 8)
    ci, grp = ci[o], grp[o]
    first = torch.ones_like(grp, dtype=torch.bool); first[1:] = grp[1:] != grp[:-1]
    idx = torch.arange(len(grp), device=dev)
    start = torch.cummax(torch.where(first, idx, torch.zeros_like(idx)), 0).values
    t = idx - start - cof                                  # rank in the group after skipping cof
    cnt = torch.zeros(int(grp.max()) + 1 if len(grp) else 1, dtype=torch.long, device=dev).index_add_(
        0, grp, torch.ones_like(grp))
    full = (t >= 0) & ((t // n + 1) * n <= cnt[grp] - cof)
    tid = grp * 1_000_000 + t // n                        # task key
    sel = torch.nonzero(full).flatten()
    if len(sel) == 0:
        return None
    key, inv = torch.unique(tid[sel], return_inverse=True)
    T = len(key)
    cells_i = torch.empty(T, n, dtype=torch.long, device=dev)
    cells_i[inv, t[sel] % n] = ci[sel]                     # into S.cells, x order within the task
    tg = torch.zeros(T, dtype=torch.long, device=dev); tg[inv] = grp[sel]
    rows = rof + tg[:, None] * m + torch.arange(m, device=dev)[None, :]
    rows = torch.where(rows < S.R, rows, torch.full_like(rows, -1))
    # per task and row: the task's cells in that row must be consecutive in their segment
    r_c = row[cells_i]                                      # (T, n)
    k_c = r_c - rows[:, :1]                                 # row slot 0..m-1
    rk = S.rank[cells_i]                                    # position in (segment, x) order
    sg = S.seg[cells_i]
    valid = torch.ones(T, dtype=torch.bool, device=dev)
    lo = torch.zeros(T, m, dtype=torch.long, device=dev); hi = torch.zeros(T, m, dtype=torch.long, device=dev)
    nxt_x = self._next_x(cells_i)                           # global sites: the next cell's x, or the segment end
    space = nxt_x - gx[cells_i]
    for k in range(m):
        inr = k_c == k
        c = inr.sum(1)
        has = c > 0
        BIGI = 1 << 40
        rmin = torch.where(inr, rk, torch.full_like(rk, BIGI)).min(1).values
        rmax = torch.where(inr, rk, torch.full_like(rk, -1)).max(1).values
        smin = torch.where(inr, sg, torch.full_like(sg, BIGI)).min(1).values
        smax = torch.where(inr, sg, torch.full_like(sg, -1)).max(1).values
        valid &= ~has | ((rmax - rmin + 1 == c) & (smin == smax))
        xmin = torch.where(inr, gx[cells_i], torch.full_like(rk, BIGI)).min(1).values
        lst = torch.where(inr, rk, torch.full_like(rk, -1)).argmax(1)
        xend = nxt_x.gather(1, lst[:, None]).squeeze(1)
        lo[:, k] = torch.where(has, xmin, torch.zeros_like(xmin))
        hi[:, k] = torch.where(has, xend, torch.zeros_like(xend))
    v = torch.nonzero(valid).flatten()
    return {'ci': cells_i[v], 'cells': S.cells[cells_i[v]], 'rows': rows[v], 'lo': lo[v], 'hi': hi[v],
            'x': gx[cells_i[v]], 'w': w[cells_i[v]], 'space': space[v], 'row0': r_c[v]}


def _next_x(self, cells_i):
    """Global site x of the next cell in the same segment after each cell, else its segment's end."""
    S = self.S
    rk = S.rank[cells_i]
    nrk = (rk + 1).clamp(max=len(S.sorted) - 1)
    nci = S.sorted[nrk]
    same = (rk + 1 < len(S.sorted)) & (S.seg[nci] == S.seg[cells_i])
    nx = S.x[nci].long() + self.roff[S.row[nci]]
    segend = S.s1[S.seg[cells_i]].long() + self.roff[S.row[cells_i]]
    return torch.where(same, nx, segend)

## 3. Every permutation, with a DP over the rows (Algorithm 2) and the x heuristic (Algorithm 3)

For one task, a permutation only fixes the **order** in which cells are placed. Cell `i` of that order goes to
the row that minimizes the cost so far, given where the previous cells went: the DP state after placing a cell
in row `j` records, per row, where the free space starts (`min_x`) and where the next cell would start if every
cell kept its own spacing (`init_x`). The x inside the row comes from **Algorithm 3**: the cell moves towards the
side where more of its connected pins are, onto the nearest of them, as far as the free space allows.

The cost during the DP is the HPWL of the cell's nets with every other pin where it is. All tasks of a batch and
all their permutations run at once: the state is a tensor (tasks x permutations x rows x rows).

In [ ]:
def _cell_data(self, tk):
    """For every task cell: its costed nets (padded to K) with the bounding box of their other pins, its pin
    offsets per candidate row (orientation as the rails need), and its connected pins' x sorted (padded to Q)."""
    S, d, dev = self.S, self.d, self.dev
    cells = tk['cells']; T, n = cells.shape; m = tk['rows'].shape[1]
    cf = cells.reshape(-1)                                  # (T*n,)
    L = len(cf)
    cnt = S.cell_ptr[cf + 1] - S.cell_ptr[cf]
    own = torch.repeat_interleave(torch.arange(L, device=dev), cnt)
    off = torch.arange(len(own), device=dev) - torch.repeat_interleave(torch.cumsum(cnt, 0) - cnt, cnt)
    pin = S.pin_by_cell[S.cell_ptr[cf][own] + off]
    net = d.p_net[pin]
    keep = S.net_ok[net]
    own, pin, net = own[keep], pin[keep], net[keep]
    E = len(S.deg)
    key, pinv = torch.unique(own * E + net, return_inverse=True)   # (cell slot, net) pairs, sorted by slot
    po, pn = key // E, key % E
    NP = len(key)
    # pair index within its cell, capped at K
    first = torch.ones(NP, dtype=torch.bool, device=dev); first[1:] = po[1:] != po[:-1]
    ii = torch.arange(NP, device=dev)
    st = torch.cummax(torch.where(first, ii, torch.zeros_like(ii)), 0).values
    kk = ii - st
    Kn = int(kk.max()) + 1 if NP else 1
    if Kn > self.K:
        raise ValueError('a cell has %d costed nets, more than K=%d' % (Kn, self.K))
    # other pins of each pair's net
    dg = S.deg[pn]
    rep = torch.repeat_interleave(torch.arange(NP, device=dev), dg)
    o2 = torch.arange(len(rep), device=dev) - torch.repeat_interleave(torch.cumsum(dg, 0) - dg, dg)
    opins = d.n_ptr[pn][rep] + o2
    oc = d.p_cell[opins]
    other = oc != cf[po[rep]]
    rep, opins, oc = rep[other], opins[other], oc[other]
    P = d.pos[oc].double() + d.size[oc].double() / 2 + d.pin_off[opins].double()
    lo = torch.full((NP, 2), INF, dtype=torch.float64, device=dev).scatter_reduce(0, rep[:, None].expand(-1, 2), P, 'amin')
    hi = torch.full((NP, 2), -INF, dtype=torch.float64, device=dev).scatter_reduce(0, rep[:, None].expand(-1, 2), P, 'amax')
    # this cell's pin offsets per candidate row (relative to the lower-left)
    rows = tk['rows'].clamp(min=0)                          # (T, m)
    rw = rows[:, None, :].expand(T, n, m).reshape(L, m)
    oj = torch.stack([S.orient_at(cf, rw[:, j]) for j in range(m)], 1)    # (L, m)
    mc = cf[own]
    offs = []
    for j in range(m):
        offs.append(_orient_offset(oj[own, j], d._p_ctr_n[pin, 0], d._p_ctr_n[pin, 1], d.size_n[mc, 0],
                                   d.size_n[mc, 1]).double())
    offs = torch.stack(offs, 1)                             # (pins, m, 2)
    pr = pinv
    omin = torch.full((NP, m, 2), INF, dtype=torch.float64, device=dev).scatter_reduce(
        0, pr[:, None, None].expand(-1, m, 2), offs, 'amin')
    omax = torch.full((NP, m, 2), -INF, dtype=torch.float64, device=dev).scatter_reduce(
        0, pr[:, None, None].expand(-1, m, 2), offs, 'amax')
    wn = S.w_net[pn]
    # padded (L, K, ...)
    D = {}
    D['lo'] = torch.full((L, Kn, 2), INF, dtype=torch.float64, device=dev); D['lo'][po, kk] = lo
    D['hi'] = torch.full((L, Kn, 2), -INF, dtype=torch.float64, device=dev); D['hi'][po, kk] = hi
    D['omin'] = torch.zeros((L, m, Kn, 2), dtype=torch.float64, device=dev); D['omin'][po, :, kk] = omin
    D['omax'] = torch.zeros((L, m, Kn, 2), dtype=torch.float64, device=dev); D['omax'][po, :, kk] = omax
    D['wn'] = torch.zeros((L, Kn), dtype=torch.float64, device=dev); D['wn'][po, kk] = wn
    # padded nets: lo = +inf, hi = -inf, offsets 0, weight 0 -> contribute 0 (guarded below)
    D['pad'] = D['wn'] == 0
    D['orient'] = oj
    # y cost per row: sum_k w (max(hi_y, Y + omax_y) - min(lo_y, Y + omin_y))
    Y = S.ys.double()[rows]                                  # (T, m)
    Yl = Y[:, None, :].expand(T, n, m).reshape(L, m)
    hy = torch.maximum(D['hi'][:, None, :, 1], Yl[:, :, None] + D['omax'][..., 1])
    ly = torch.minimum(D['lo'][:, None, :, 1], Yl[:, :, None] + D['omin'][..., 1])
    yc = torch.where(D['pad'][:, None, :], torch.zeros_like(hy), (hy - ly) * D['wn'][:, None, :]).sum(2)
    D['ycost'] = torch.where(tk['rows'][:, None, :].expand(T, n, m).reshape(L, m) >= 0, yc, torch.full_like(yc, INF))
    # connected pins' x, sorted, padded with +inf (at most Q, the nearest to the cell)
    cxo = d.pos[cf, 0].double() + d.size[cf, 0].double() / 2
    ow = po[rep]
    px = P[:, 0]
    dist = (px - cxo[ow]).abs()
    o3 = torch.argsort(ow.double() * 1e12 + dist)
    ow, px = ow[o3], px[o3]
    f3 = torch.ones_like(ow, dtype=torch.bool); f3[1:] = ow[1:] != ow[:-1]
    i3 = torch.arange(len(ow), device=dev)
    s3 = torch.cummax(torch.where(f3, i3, torch.zeros_like(i3)), 0).values
    q = i3 - s3
    k3 = q < self.Q
    X = torch.full((L, self.Q), INF, dtype=torch.float64, device=dev)
    X[ow[k3], q[k3]] = px[k3]
    D['px'] = torch.sort(X, 1).values
    D['npx'] = torch.isfinite(D['px']).sum(1)
    for k in ('lo', 'hi', 'omin', 'omax', 'wn', 'ycost', 'px'):
        D[k] = D[k].to(self.dt)
    return D


def _perms(self, n):
    if n not in self.perms:
        self.perms[n] = torch.tensor(list(itertools.permutations(range(n))), device=self.dev)
    return self.perms[n]

In [ ]:
def _dp(self, tk, D, b0, b1):
    """Algorithm 2 for tasks b0:b1 and every permutation. Returns (cost (B, P) DP cost, row slot (B, P, n), x
    (B, P, n) global sites), indexed by the task's cell slot (x order), not by processing order."""
    dev = self.dev
    Pm = self._perms(tk['cells'].shape[1])
    P, n = Pm.shape
    m = tk['rows'].shape[1]
    B = b1 - b0
    dt = self.dt
    lo = tk['lo'][b0:b1].to(dt); hi = tk['hi'][b0:b1].to(dt)            # (B, m)
    w = tk['w'][b0:b1].to(dt); sp = tk['space'][b0:b1].to(dt)           # (B, n)
    base = torch.arange(b0, b1, device=dev)[:, None] * n               # cell-slot row in D
    # state: (B, P, Kst, m) min_x / init_x per row; cost (B, P, Kst)
    mn = lo[:, None, None, :].expand(B, P, 1, m).clone()
    ini = mn.clone()
    cost = torch.zeros(B, P, 1, dtype=dt, device=dev)
    prevs, xs = [], []
    ar_m = torch.arange(m, device=dev)
    for i in range(n):
        s = Pm[:, i]                                                    # (P,) cell slot processed at step i
        L = (base + s[None, :])                                         # (B, P) row into D
        wc = w[:, s]; sc = sp[:, s]                                      # (B, P)
        # candidate (k: previous state, j: row): min_x = mn[k, j]; init = ini[k, j]
        mnkj = mn                                                       # (B, P, Kst, m)
        inkj = ini
        hij = hi[:, None, None, :]
        feas = hij - mnkj >= wc[:, :, None, None]
        maxp = torch.minimum(inkj + sc[:, :, None, None], hij.expand_as(inkj))
        x1 = torch.minimum(inkj, maxp - wc[:, :, None, None])
        x2 = self._adjust(D, L, x1, mnkj, maxp, wc)
        feas &= (x2 >= mnkj) & (x2 + wc[:, :, None, None] <= maxp)
        c = cost[:, :, :, None] + self._xcost(D, L, x2) + D['ycost'][L][:, :, None, :]
        c = torch.where(feas, c, torch.full_like(c, INF))
        bc, bk = c.min(2)                                               # over previous states -> (B, P, m)
        bx = x2.gather(2, bk[:, :, None, :]).squeeze(2)
        # new states, one per row j: copy state bk[j], then update row j
        nmn = mn.gather(2, bk[..., None].expand(B, P, m, m))            # (B, P, m(state j), m(row))
        nin = ini.gather(2, bk[..., None].expand(B, P, m, m))
        eye = ar_m[:, None] == ar_m[None, :]                            # state j updates row j
        nmn = torch.where(eye, (bx + wc[:, :, None])[..., None].expand_as(nmn), nmn)
        nin = torch.where(eye, (nin + sc[:, :, None, None]), nin)
        mn, ini, cost = nmn, nin, bc
        prevs.append(bk); xs.append(bx)
    # best final state, backtrack
    fc, j = cost.min(2)                                                 # (B, P)
    rows = torch.empty(B, P, n, dtype=torch.long, device=dev)
    xx = torch.empty(B, P, n, dtype=dt, device=dev)
    for i in range(n - 1, -1, -1):
        s = Pm[:, i][None, :].expand(B, P)
        rows.scatter_(2, s[..., None], j[..., None])
        xx.scatter_(2, s[..., None], xs[i].gather(2, j[..., None]))
        j = prevs[i].gather(2, j[..., None]).squeeze(2)
    return fc, rows, xx

In [ ]:
def _xcost(self, D, L, xs):
    """HPWL in x of each candidate: cells D-rows L (B, P) at global sites xs (B, P, Kst, m) in row j (offsets of
    row j). -> (B, P, Kst, m)"""
    X = self.X0 + xs * self.sw                                          # um, lower-left
    lo = D['lo'][L][..., 0]; hi = D['hi'][L][..., 0]                    # (B, P, K)
    omin = D['omin'][L][..., 0]; omax = D['omax'][L][..., 0]            # (B, P, m, K)
    wn = D['wn'][L]
    a = torch.maximum(hi[:, :, None, None, :], X[..., None] + omax[:, :, None, :, :])
    b = torch.minimum(lo[:, :, None, None, :], X[..., None] + omin[:, :, None, :, :])
    v = torch.where(D['pad'][L][:, :, None, None, :], torch.zeros_like(a), (a - b) * wn[:, :, None, None, :])
    return v.sum(-1)


def _adjust(self, D, L, x1, lo, hi, wc):
    """Algorithm 3: towards the side of the cell's connected pins with more pins (ties: left), to the nearest pin
    on that side (cell centre on it, rounded to a site), clamped to [lo, hi - w]."""
    px = D['px'][L]                                                     # (B, P, Q) sorted, +inf padded
    npx = D['npx'][L]                                                   # (B, P)
    B, P, Ks, m = x1.shape
    cx = self.X0 + (x1 + wc[:, :, None, None] / 2) * self.sw            # cell centre, um
    q = cx.reshape(B, P, Ks * m).contiguous()
    nl = torch.searchsorted(px.contiguous(), q, right=False)            # pins strictly left of the centre
    nle = torch.searchsorted(px.contiguous(), q, right=True)            # pins left or at the centre
    nr = npx[..., None] - nle
    left = px.gather(2, (nl - 1).clamp(min=0))
    right = px.gather(2, nle.clamp(max=px.shape[2] - 1))
    use_l = (nl >= nr) & (nl > 0)
    use_r = ~use_l & (nr > 0)
    tgt = torch.where(use_l, left, torch.where(use_r, right, q))
    x2 = torch.round((tgt - self.X0) / self.sw - wc.reshape(B, P, 1) / 2).reshape(B, P, Ks, m)
    x2 = torch.where(use_l.reshape(B, P, Ks, m) | use_r.reshape(B, P, Ks, m), x2, x1)
    return torch.minimum(torch.maximum(x2, lo), hi - wc[:, :, None, None])

## 4. The exact cost of a permutation (Algorithm 1, line 7)

With all cells of a task placed, the permutation is scored exactly: Attar's `_DP.cost` gives the HPWL of every
net the task touches; for the second stage the same nets are scored by their **rectilinear MST** (section 1's
`rmst`). Only the `top` permutations with the best DP cost (8) are scored exactly.

In [ ]:
def cost_mst(S, mv_cells, mv_x, mv_y, mv_o):
    """As _DP.cost, the nets' rectilinear MST length (HPWL for 2-3 pins) instead of their HPWL. -> (C,)"""
    d, dev = S.d, S.dev
    C, Mm = mv_cells.shape
    valid = mv_cells >= 0
    cc = mv_cells.clamp(min=0)
    cnt = torch.where(valid, S.cell_ptr[cc + 1] - S.cell_ptr[cc], torch.zeros_like(cc)).reshape(-1)
    slot = torch.repeat_interleave(torch.arange(C * Mm, device=dev), cnt)
    off = torch.arange(len(slot), device=dev) - torch.repeat_interleave(torch.cumsum(cnt, 0) - cnt, cnt)
    pin = S.pin_by_cell[S.cell_ptr[cc.reshape(-1)[slot]] + off]
    net = d.p_net[pin]
    keep = S.net_ok[net]
    cand = slot[keep] // Mm; net = net[keep]
    if len(cand) == 0:
        return torch.zeros(C, dtype=torch.float64, device=dev)
    E = len(S.deg)
    key = torch.unique(cand * E + net)
    pc, pn = key // E, key % E
    dg = S.deg[pn]
    rep = torch.repeat_interleave(torch.arange(len(key), device=dev), dg)
    po = torch.arange(len(rep), device=dev) - torch.repeat_interleave(torch.cumsum(dg, 0) - dg, dg)
    pins = d.n_ptr[pn][rep] + po
    pcell = d.p_cell[pins]
    mvc = mv_cells[pc[rep]]
    hit = mvc == pcell[:, None]
    anyhit = hit.any(1)
    j = hit.long().argmax(1)
    ci = pc[rep]
    nan = torch.full((len(pins),), float('nan'), dtype=torch.float64, device=dev)
    sx = torch.where(anyhit, mv_x[ci, j].double(), nan)
    sy = torch.where(anyhit, mv_y[ci, j].double(), nan)
    so = torch.where(anyhit, mv_o[ci, j], torch.zeros_like(j))
    Pp = S.pin_pos_with(pins, pcell, sx, sy, so)
    ptr = torch.cat([torch.zeros(1, dtype=torch.long, device=dev), torch.cumsum(dg, 0)])
    lo = torch.full((len(key), 2), INF, dtype=torch.float64, device=dev).scatter_reduce(0, rep[:, None].expand(-1, 2), Pp, 'amin')
    hi = torch.full((len(key), 2), -INF, dtype=torch.float64, device=dev).scatter_reduce(0, rep[:, None].expand(-1, 2), Pp, 'amax')
    hp = (hi - lo).sum(1)
    big = dg >= 4
    wl = hp.clone()
    if bool(big.any()):
        r = rmst(Pp, dg, ptr, big)
        wl = torch.where(big, r, hp)
    return torch.zeros(C, dtype=torch.float64, device=dev).index_add_(0, pc, wl * S.w_net[pn])


def exact(self, mv_cells, x, y, o, kind):
    if kind == 'hpwl':
        return self.S.cost(mv_cells, x, y, o, xonly=False)
    return cost_mst(self.S, mv_cells, x, y, o)

## 5. Independent task sets, and one SROA pass

Tasks that share no net can be applied together, and their gains add up exactly. The sets are grown greedily as
in the paper (on every net the free task with the lowest random priority wins; a task touching a taken net waits
for a later set). A pass then runs every set: all permutations, the DP, the exact cost, and the moves that gain.

In [ ]:
def _sets(self, cells):
    """Independent task sets (paper III-B, greedy): each set is a maximal group of tasks sharing no costed net,
    grown in rounds (on every net the free task of lowest random priority wins; a task touching a taken net waits for a later
    set). The (task, net) pairs are computed once: they do not depend on the positions. -> (set per task, sets)"""
    S, dev = self.S, self.dev
    T = len(cells)
    E = len(S.deg)
    pc, pn = S.nets_of(cells)
    g = torch.Generator(device='cpu').manual_seed(self.seed)
    prio = torch.randperm(T, generator=g).to(dev)          # random priorities: few rounds per set (Luby)
    color = torch.full((T,), -1, dtype=torch.long, device=dev)
    remaining = torch.ones(T, dtype=torch.bool, device=dev)
    k = 0
    while bool(remaining.any()):
        taken = torch.zeros(E, dtype=torch.bool, device=dev)
        free = remaining.clone()
        while bool(free.any()):
            fp = free[pc]
            own = torch.full((E,), T, dtype=torch.long, device=dev).scatter_reduce(0, pn[fp], prio[pc[fp]], 'amin')
            lose = torch.zeros(T, dtype=torch.bool, device=dev)
            lose[pc[fp & (own[pn] != prio[pc])]] = True
            win = free & ~lose
            color[win] = k
            remaining &= ~win
            taken[pn[win[pc]]] = True
            blocked = torch.zeros(T, dtype=torch.bool, device=dev)
            blocked[pc[taken[pn]]] = True
            free &= ~win & ~blocked
        k += 1
    return color, k


def run(self, n, m, rof, cof, kind='hpwl'):
    """One SROA pass for (n, m, rof, cof); kind: the exact cost ('hpwl' or 'mst'). Returns statistics."""
    S, d, dev = self.S, self.d, self.dev
    t0 = time.time()
    tk = self.tasks(n, m, rof, cof)
    st = {'n': n, 'm': m, 'rof': rof, 'cof': cof, 'kind': kind, 'tasks': 0, 'applied': 0, 'gain': 0.0, 'sets': 0}
    if tk is None:
        return st
    T = len(tk['cells'])
    st['tasks'] = T
    color, nsets = self._sets(tk['cells'])
    st['sets'] = nsets
    Pm = self._perms(n)
    P = len(Pm)
    for k in range(nsets):
        cur_set = torch.nonzero(color == k).flatten()
        sub = {k: v[cur_set] for k, v in tk.items()}
        D = self._cell_data(sub)
        B = len(cur_set)
        best_x = torch.empty(B, n, dtype=torch.float64, device=dev)
        best_r = torch.empty(B, n, dtype=torch.long, device=dev)
        gain = torch.zeros(B, dtype=torch.float64, device=dev)
        step = max(1, self.chunk * 720 // max(P, 1) // max(1, n * m * m // 6))
        for b0 in range(0, B, step):
            b1 = min(B, b0 + step)
            fc, rows, xx = self._dp(sub, D, b0, b1)                    # (b, P), (b, P, n)
            bb = b1 - b0
            kk = P if self.top is None else min(self.top, P)
            cand = torch.topk(fc, kk, dim=1, largest=False).indices    # (b, kk)
            cfin = torch.isfinite(fc.gather(1, cand))
            r_sel = rows.gather(1, cand[..., None].expand(-1, -1, n))  # (b, kk, n) row slot
            x_sel = xx.gather(1, cand[..., None].expand(-1, -1, n)).double()
            rid = sub['rows'][b0:b1].gather(1, r_sel.reshape(bb, -1).clamp(min=0)).reshape(bb, kk, n)
            cells = sub['cells'][b0:b1]
            mvc = cells[:, None, :].expand(bb, kk, n).reshape(-1, n)
            xum = (self.X0 + x_sel * self.sw).reshape(-1, n)
            yum = S.ys.double()[rid].reshape(-1, n)
            o = S.orient_at(mvc.reshape(-1), rid.reshape(-1)).view(-1, n)
            c = self.exact(mvc, xum, yum, o, kind).view(bb, kk)
            c = torch.where(cfin, c, torch.full_like(c, INF))
            if S.pdn is not None:
                ok = S.pdn_ok(mvc.reshape(-1), xum.reshape(-1), yum.reshape(-1), o.reshape(-1)).view(bb, kk, n).all(2)
                c = torch.where(ok, c, torch.full_like(c, INF))
            cb, ib = c.min(1)
            curc = self.exact(cells, d.pos[cells, 0].double(), d.pos[cells, 1].double(), d.orient[cells].long(), kind)
            gain[b0:b1] = torch.where(torch.isfinite(cb), curc - cb, torch.zeros_like(cb))
            best_x[b0:b1] = x_sel[torch.arange(bb, device=dev), ib]
            best_r[b0:b1] = rid[torch.arange(bb, device=dev), ib]
        good = gain > 1e-9
        if bool(good.any()):
            g = torch.nonzero(good).flatten()
            cells = sub['cells'][g].reshape(-1)
            r = best_r[g].reshape(-1)
            xum = self.X0 + best_x[g].reshape(-1) * self.sw
            S.apply(cells, xum, S.ys.double()[r], S.orient_at(cells, r))
            st['applied'] += len(g); st['gain'] += float(gain[g].sum())
    S.refresh()
    st['time'] = time.time() - t0
    return st


for name in ('__init__', 'tasks', '_next_x', '_cell_data', '_perms', '_dp', '_xcost', '_adjust', 'exact', '_sets', 'run'):
    setattr(SROA, name, globals()[name])

**Check.** One pass on a copy of the start: the gains SROA reports must equal the drop of the nets' HPWL, and the
placement must stay legal.

In [ ]:
dc = placed()
A = SROA(dc, verbose=False)
h0 = hpwl_costed(dc, A.S)
st = A.run(4, 2, 0, 0, kind='hpwl')
print('tasks %d, sets %d, applied %d: gain %.3f um, HPWL drop %.3f um, legal %s, %.1f s' % (
    st['tasks'], st['sets'], st['applied'], st['gain'], h0 - hpwl_costed(dc, A.S),
    Evaluator(dc).legality()['legal'], st['time']))

## 6. Independent set matching

The paper keeps ABCDPlace's ISM before SROA in each stage: cells of the same width that share no net are grouped
by location (up to 64), and each group is re-assigned to its own slots by a min-cost assignment of the HPWL.

In [ ]:
ORIENTS = (0, 1, 4, 5)            # the orientations single-row cells take (N, S, FN, FS)


class ISM:
    """Independent set matching (methods below)."""

def __init__(self, sroa, group=64, row_bin=16, site_bin=400):
    self.A, self.S, self.d, self.dev = sroa, sroa.S, sroa.d, sroa.dev
    self.group, self.row_bin, self.site_bin = group, row_bin, site_bin


def _data(self, cells):
    """Per cell: its costed nets (padded) with the box of their other pins, and its pin offsets in each orientation."""
    S, d, dev = self.S, self.d, self.dev
    L = len(cells)
    cnt = S.cell_ptr[cells + 1] - S.cell_ptr[cells]
    own = torch.repeat_interleave(torch.arange(L, device=dev), cnt)
    off = torch.arange(len(own), device=dev) - torch.repeat_interleave(torch.cumsum(cnt, 0) - cnt, cnt)
    pin = S.pin_by_cell[S.cell_ptr[cells][own] + off]
    net = d.p_net[pin]
    keep = S.net_ok[net]
    own, pin, net = own[keep], pin[keep], net[keep]
    E = len(S.deg)
    key, pinv = torch.unique(own * E + net, return_inverse=True)
    po, pn = key // E, key % E
    NP = len(key)
    first = torch.ones(NP, dtype=torch.bool, device=dev); first[1:] = po[1:] != po[:-1]
    ii = torch.arange(NP, device=dev)
    kk = ii - torch.cummax(torch.where(first, ii, torch.zeros_like(ii)), 0).values
    K = int(kk.max()) + 1 if NP else 1
    dg = S.deg[pn]
    rep = torch.repeat_interleave(torch.arange(NP, device=dev), dg)
    o2 = torch.arange(len(rep), device=dev) - torch.repeat_interleave(torch.cumsum(dg, 0) - dg, dg)
    opins = d.n_ptr[pn][rep] + o2
    oc = d.p_cell[opins]
    other = oc != cells[po[rep]]
    rep, opins, oc = rep[other], opins[other], oc[other]
    P = d.pos[oc].double() + d.size[oc].double() / 2 + d.pin_off[opins].double()
    lo = torch.full((NP, 2), INF, dtype=torch.float64, device=dev).scatter_reduce(0, rep[:, None].expand(-1, 2), P, 'amin')
    hi = torch.full((NP, 2), -INF, dtype=torch.float64, device=dev).scatter_reduce(0, rep[:, None].expand(-1, 2), P, 'amax')
    mc = cells[own]
    offs = torch.stack([_orient_offset(torch.full_like(pin, o), d._p_ctr_n[pin, 0], d._p_ctr_n[pin, 1],
                                       d.size_n[mc, 0], d.size_n[mc, 1]).double() for o in ORIENTS], 1)
    nO = len(ORIENTS)
    omin = torch.full((NP, nO, 2), INF, dtype=torch.float64, device=dev).scatter_reduce(
        0, pinv[:, None, None].expand(-1, nO, 2), offs, 'amin')
    omax = torch.full((NP, nO, 2), -INF, dtype=torch.float64, device=dev).scatter_reduce(
        0, pinv[:, None, None].expand(-1, nO, 2), offs, 'amax')
    D = {'lo': torch.full((L, K, 2), INF, dtype=torch.float64, device=dev),
         'hi': torch.full((L, K, 2), -INF, dtype=torch.float64, device=dev),
         'omin': torch.zeros((L, nO, K, 2), dtype=torch.float64, device=dev),
         'omax': torch.zeros((L, nO, K, 2), dtype=torch.float64, device=dev),
         'wn': torch.zeros((L, K), dtype=torch.float64, device=dev)}
    D['lo'][po, kk] = lo; D['hi'][po, kk] = hi
    D['omin'][po, :, kk] = omin; D['omax'][po, :, kk] = omax
    D['wn'][po, kk] = S.w_net[pn]
    return D


def _cost(self, D, gi, slot_xy, slot_o):
    """C[g, i, j]: cell gi[g, i] at slot j of its group (lower-left slot_xy[g, j], orientation index slot_o[g, i, j])."""
    lo, hi, wn = D['lo'][gi], D['hi'][gi], D['wn'][gi]                 # (G, n, K, 2), (G, n, K)
    G, n = gi.shape
    oi = slot_o[..., None, None, None].expand(G, n, n, 1, lo.shape[2], 2)
    omin = D['omin'][gi][:, :, None].expand(G, n, n, -1, -1, -1).gather(3, oi).squeeze(3)   # (G, n, n, K, 2)
    omax = D['omax'][gi][:, :, None].expand(G, n, n, -1, -1, -1).gather(3, oi).squeeze(3)
    p = slot_xy[:, None, :, None, :]                                     # (G, 1, n, 1, 2)
    a = torch.maximum(hi[:, :, None], p + omax)
    b = torch.minimum(lo[:, :, None], p + omin)
    v = torch.where(wn[:, :, None, :, None] > 0, a - b, torch.zeros_like(a)) * wn[:, :, None, :, None]
    return v.sum((3, 4))                                                # (G, n, n)


def run(self, max_passes=3, tol=1e-4):
    """ISM passes until a pass gains less than tol of the costed HPWL. Returns statistics."""
    A, S, d, dev = self.A, self.S, self.d, self.dev
    t0 = time.time()
    st = {'passes': 0, 'moved': 0, 'gain': 0.0}
    S.refresh()
    color, ncol = A._sets(S.cells[:, None])                  # cells sharing no costed net
    for _ in range(max_passes):
        h0 = hpwl_costed(d, S)
        gained = 0.0
        for c in range(ncol):
            S.refresh()
            ci = torch.nonzero(color == c).flatten()          # into S.cells
            if len(ci) < 2:
                continue
            w = S.w[ci].long(); r = S.row[ci]; gx = S.x[ci].long() + A.roff[r]
            key = ((w * 100000 + r // self.row_bin) * 100000 + gx // self.site_bin) * 100000 + gx
            o = torch.argsort(key.double())
            ci, w, r, gx = ci[o], w[o], r[o], gx[o]
            bkey = (w * 100000 + r // self.row_bin) * 100000 + gx // self.site_bin
            first = torch.ones(len(ci), dtype=torch.bool, device=dev); first[1:] = bkey[1:] != bkey[:-1]
            idx = torch.arange(len(ci), device=dev)
            start = torch.cummax(torch.where(first, idx, torch.zeros_like(idx)), 0).values
            pos_in = idx - start
            gid = torch.cumsum(first.long(), 0) * 100000 + pos_in // self.group
            gk, ginv = torch.unique(gid, return_inverse=True)
            cntg = torch.bincount(ginv)
            ok = cntg[ginv] >= 2
            ci, r, gx, ginv = ci[ok], r[ok], gx[ok], ginv[ok]
            if len(ci) == 0:
                continue
            gk2, ginv = torch.unique(ginv, return_inverse=True)
            cntg = torch.bincount(ginv)
            n = int(cntg.max())
            G = len(gk2)
            slot = torch.zeros_like(ginv)
            st_ = torch.cumsum(cntg, 0) - cntg
            slot = torch.arange(len(ginv), device=dev) - st_[ginv]
            gi = torch.full((G, n), -1, dtype=torch.long, device=dev); gi[ginv, slot] = torch.arange(len(ci), device=dev)
            valid = gi >= 0
            cells = S.cells[ci]
            D = self._data(cells)
            gic = gi.clamp(min=0)
            sxy = torch.stack([A.X0 + gx.double() * A.sw, S.ys.double()[r]], 1)[gic]   # (G, n, 2) slots
            srow = r[gic]
            # orientation of cell i at slot j (rails), as an index into ORIENTS
            ce = cells[gic][:, :, None].expand(G, n, n).reshape(-1)
            rj = srow[:, None, :].expand(G, n, n).reshape(-1)
            oij = S.orient_at(ce, rj).view(G, n, n)
            oidx = torch.zeros_like(oij)
            for k, ov in enumerate(ORIENTS):
                oidx = torch.where(oij == ov, torch.full_like(oidx, k), oidx)
            C = torch.empty(G, n, n, dtype=torch.float64, device=dev)
            step = max(1, 4_000_000 // (n * n * max(1, D['lo'].shape[1])))
            for g0 in range(0, G, step):
                g1 = min(G, g0 + step)
                C[g0:g1] = self._cost(D, gic[g0:g1], sxy[g0:g1], oidx[g0:g1])
            pad = ~(valid[:, :, None] & valid[:, None, :])
            C = torch.where(pad, torch.where(torch.eye(n, dtype=torch.bool, device=dev)[None], torch.zeros_like(C),
                                             torch.full_like(C, 1e18)), C)
            cur = torch.diagonal(C, dim1=1, dim2=2).sum(1)
            Cn = C.cpu().numpy()
            perm = np.empty((G, n), dtype=np.int64)
            best = np.empty(G)
            for g in range(G):
                rr, cc = linear_sum_assignment(Cn[g])
                perm[g, rr] = cc
                best[g] = Cn[g][rr, cc].sum()
            perm = torch.from_numpy(perm).to(dev); best = torch.from_numpy(best).to(dev)
            gain = cur - best
            good = gain > 1e-9
            if not bool(good.any()):
                continue
            gg = torch.nonzero(good).flatten()
            src = gi[gg]                                         # (k, n) cell index (into ci), -1 pad
            dst = perm[gg]                                       # slot taken by each member
            m = (src >= 0) & (dst != torch.arange(n, device=dev)[None, :])
            mc = cells[src[m]]
            tgt = gi[gg][:, :][torch.arange(len(gg), device=dev)[:, None].expand(-1, n)[m], dst[m]]
            nx = A.X0 + gx[tgt].double() * A.sw; ny = S.ys.double()[r[tgt]]
            no = S.orient_at(mc, r[tgt])
            okp = S.pdn_ok(mc, nx, ny, no) if S.pdn is not None else torch.ones(len(mc), dtype=torch.bool, device=dev)
            # a group whose any moved cell hits the PDN is skipped whole (its permutation stays consistent)
            grp_of = torch.arange(len(gg), device=dev)[:, None].expand(-1, n)[m]
            bad = torch.zeros(len(gg), dtype=torch.bool, device=dev); bad[grp_of[~okp]] = True
            keepm = ~bad[grp_of]
            if bool(keepm.any()):
                S.apply(mc[keepm], nx[keepm], ny[keepm], no[keepm])
                gained += float(gain[gg][~bad].sum()); st['moved'] += int(keepm.sum())
        st['passes'] += 1
        st['gain'] += gained
        S.refresh()
        if gained < tol * h0:
            break
    st['time'] = time.time() - t0
    return st


for name in ('__init__', '_data', '_cost', 'run'):
    setattr(ISM, name, globals()[name])

## 7. The flow (paper Fig. 2 and IV-A)

Stage 1 (HPWL): ISM, then SROA with 4-cell tasks. Stage 2 (MST wirelength): ISM, then SROA with 6-cell tasks,
then with 4-cell tasks. Each SROA round runs 4 iterations over 3, 2, 3 and 2 rows, each iteration every row and
column offset the paper lists.

In [ ]:
COMBOS = {3: [(r, c) for r in (0, 1, 2) for c in (0, 3)], 2: [(r, c) for r in (0, 1) for c in (0, 2, 4)]}
ITER_M = (3, 2, 3, 2)
FLOWS = {'paper': ['ism', ('hpwl', 4), 'ism', ('mst', 'n2'), ('mst', 4)]}


def run_flow(d, flow='paper', n2=6, top=8, ism=True, log=print):
    A = SROA(d, top=top, verbose=False)
    rounds = []
    for step in FLOWS[flow]:
        if step == 'ism':
            if ism:
                st = ISM(A).run()
                st['hpwl_costed'] = hpwl_costed(d, A.S); st['mstwl'] = mstwl(d, A.S)
                log('  ISM: moved %d gain %.1f  HPWL %.1f  MSTWL %.1f  (%.1f s)' % (
                    st['moved'], st['gain'], st['hpwl_costed'], st['mstwl'], st['time']))
                rounds.append({'kind': 'ism', **st})
            continue
        kind, n = step
        n = n2 if n == 'n2' else n
        r = {'kind': kind, 'n': n, 'iters': []}
        t0 = time.time()
        for m in ITER_M:
            it = {'m': m, 'passes': []}
            for rof, cof in COMBOS[m]:
                st = A.run(n, m, rof, cof, kind=kind)
                it['passes'].append(st)
            it['hpwl_costed'] = hpwl_costed(d, A.S); it['mstwl'] = mstwl(d, A.S)
            it['applied'] = sum(p['applied'] for p in it['passes'])
            log('  %s n=%d m=%d: applied %d  HPWL %.1f  MSTWL %.1f  (%.1f s)' % (
                kind, n, m, it['applied'], it['hpwl_costed'], it['mstwl'], sum(p['time'] for p in it['passes'])))
            r['iters'].append(it)
        r['time'] = time.time() - t0
        rounds.append(r)
    return rounds, A

## 8. Run SROA, and compare

From the same start: Attar's own detailed placer (`naive.detailed_place`, a batched port of OpenROAD's moves:
global swap, vertical swap, reordering, flipping), and SROA.

In [ ]:
t = time.time()
rounds, A = run_flow(d, 'paper', n2=6, top=8)
t_sroa = time.time() - t
sroa = measure(d, A.S)

da = placed()
t = time.time()
naive.detailed_place(da, verbose=False)
t_adp = time.time() - t
adp = measure(da, _DP(da))


def gain(r, k):
    return 100 * (start[k] - r[k]) / start[k]


print('%-26s %8s %8s %8s %8s %8s %7s' % ('improvement over the start', 'GR WL', 'GR vias', 'HPWL', 'MSTWL', 'time', 'legal'))
for name, r, s in (('Attar naive.detailed_place', adp, t_adp), ('SROA (paper flow)', sroa, t_sroa)):
    print('%-26s %7.2f%% %7.2f%% %7.2f%% %7.2f%% %8s %7s' % (name, gain(r, 'gr_wl'), gain(r, 'gr_vias'), gain(r, 'hpwl'),
                                                         gain(r, 'mstwl'), '%.0f s' % s if s else '-', r['legal']))

In [ ]:
import matplotlib.pyplot as plt

hist = [('start', start['hpwl_costed'], start['mstwl'])]
for r in rounds:
    if r['kind'] == 'ism':
        hist.append(('ISM', r['hpwl_costed'], r['mstwl']))
    else:
        for it in r['iters']:
            hist.append(('%s n=%d m=%d' % ('HPWL' if r['kind'] == 'hpwl' else 'MST', r['n'], it['m']), it['hpwl_costed'], it['mstwl']))
x = np.arange(len(hist))
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.plot(x, [100 * (h[1] / start['hpwl_costed'] - 1) for h in hist], marker='o', color='#d9622b', label='HPWL')
ax.plot(x, [100 * (h[2] / start['mstwl'] - 1) for h in hist], marker='s', color='#2a78d6', label='MST wirelength')
ax.axhline(0, color='#888', lw=1)
ax.set_xticks(x); ax.set_xticklabels([h[0] for h in hist], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('change from the start (%)'); ax.grid(alpha=.3); ax.legend(frameon=False)
ax.set_title('SROA on ispd19_test1: HPWL and MST wirelength after each step')
plt.tight_layout(); plt.show()

## Notes

* **From Attar's placement** (this notebook, test1 on an RTX 4090, 45 s end to end): SROA shortens the
  global-routed wirelength by 7 to 12 % and the vias by 7 to 10 % in about 40 s, against about 2 % and 1 % for
  `naive.detailed_place` (the range: Attar's global placement is not bit-for-bit reproducible, so the start
  differs from run to run). Attar's global placement leaves more room for detailed placement than DREAMPlace's,
  hence the larger numbers than below.
* **The paper's setting, all ten designs** (DREAMPlace GP + LG as the start, run separately; mean over the 8
  designs every placer finished):

  | improvement over the DREAMPlace start | GR wirelength | GR vias | time per design |
  |---|---:|---:|---:|
  | Attar `naive.detailed_place` | 2.05 % | 1.20 % | 1-10 s |
  | **SROA** | **3.34 %** | **3.10 %** | 50-535 s |

  ABCDPlace (DREAMPlace's detailed placer) reaches 1.33 % / 0.92 % there. The paper, with Innovus detailed
  routing: SROA 2.63 % / 2.47 %, ABCDPlace 1.28 % / 1.44 %. SROA buys quality with time: minutes where the
  others take seconds.
* **Where the gain comes from:** the MST stage. HPWL in both stages loses most of it (test1: 1.1 % instead of
  5.0 % GR wirelength); MST in both stages does about as well as HPWL then MST. ISM adds little.
* The ruler differs: Attar's global router here, Innovus' detailed router in the paper. Both reward the same
  thing (shorter, simpler nets), and the ranking of the placers is what is compared.
* Differences from the paper: the exact cost is computed for the 8 best permutations of the DP (all in the
  paper); ISM uses HPWL in both stages; the code is PyTorch (plus one Triton kernel), not CUDA C++.
* Other designs: `attar.load('after_fp/ispd2019/test2')` ... `test10`.